# Pre-Training Verification Checklist

**Status:** ✓ All A100 optimizations implemented and ready to train

In [1]:
print("Hello")

Hello


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Neur — Sequential Colab Pipeline

Run each cell in order. Every step reads from and writes to Google Drive so you can stop and resume between steps.

- **Step 0** — clone repo, install deps, mount Drive.
- **Step 1** — download Samanantar (2M pairs) + FLORES-200 to `Drive/neur/raw_data/`.
- **Step 2** — clean and split the raw data into `Drive/neur/processed_data/`.
- **Step 3** — train the sinusoidal model and evaluate on FLORES-200. Outputs in `Drive/neur/outputs/`.


In [19]:
!git pull https://github.com/thenileshmishra/AS-RoPE.git

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 7 (delta 3), reused 7 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 6.15 KiB | 2.05 MiB/s, done.
From https://github.com/thenileshmishra/AS-RoPE
 * branch            HEAD       -> FETCH_HEAD
Updating 056ece0..3f0bea0
Fast-forward
 notebooks/pipeline.ipynb     | 190 ++++++++++++++++++------
 pipeline/step2_preprocess.py | 336 +++++++++++++++++++++++++++++++++++--------
 requirements.txt             |   1 +
 3 files changed, 425 insertions(+), 102 deletions(-)


In [22]:
!ls

docs  notebooks  pipeline  README.md  requirements.txt	src


In [13]:
import os, sys
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
sys.path.insert(0, '/content/neur')

from pipeline import paths
paths.ensure_dirs()
print(paths.summary())

PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics


## Step 1 — Download raw datasets to Google Drive

Downloads up to 2M Samanantar pairs and the full FLORES-200 devtest split.
Safe to re-run — existing files are skipped unless `--force` is passed.

In [17]:
!python -m pipeline.step1_download --force

[step1] paths:
PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics
[step1] loading ai4bharat/samanantar/hi ...
[step1] samanantar total rows: 10,125,706
[step1] using all 10,125,706 rows
[step1] detected hi=tgt en=src translation_dict=False
[step1] progress: 99,999 pairs written...
[step1] progress: 199,999 pairs written...
[step1] progress: 299,999 pairs written...
[step1] progress: 399,999 pairs written...
[step1] progress: 499,999 pairs written...
[step1] progress: 599,999 pairs written...
[step1] progress: 699,999 pairs written...
[step1] progress: 799,999 pairs writt

## Step 2 — Clean and split the raw data

Reads `raw_data/samanantar/samanantar_hi_en.tsv`, applies cleaning filters
(length, length-ratio, script, dedupe), and writes deterministic train/val/test
splits to `processed_data/`.

In [ ]:
!python -m pipeline.step2_preprocess

[step2] streaming raw pairs from /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
2026-04-11 13:25:13.802969: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775913914.546110   14898 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775913914.661792   14898 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775913915.929431   14898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775913915.929489   14898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target m

In [ ]:
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints
!cat /content/drive/MyDrive/neur/outputs/metrics/metrics.json

## Step 3 — Train (sinusoidal) + evaluate on FLORES-200

Trains the GPT model with sinusoidal positional encoding, saving checkpoints under `outputs/checkpoints/`, training logs under `outputs/logs/`, and BLEU/chrF under `outputs/metrics/`.

In [ ]:
!python -m pipeline.step3_train \
  --use-bf16 --use-compile --pin-memory --dataloader-workers 2 \
  --grad-accum 8 --batch-size 64 --num-steps 100000

## Step 3b: Verify Tokenizer Fix (AI4Bharat + SEP ≠ EOS)

Confirms that the dedicated `<sep>` token is distinct from `</s>` (EOS),
and compares tokenization efficiency between GPT-2 and IndicBART on Hindi text.

## Verify outputs on Drive

In [ ]:
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints
!cat /content/drive/MyDrive/neur/outputs/metrics/metrics.json

In [ ]:
from src.tokenizer_utils import build_mt_tokenizer, verify_token_ids
from transformers import AutoTokenizer

# --- 1. Build the IndicBART tokenizer with dedicated <sep> ---
tok = build_mt_tokenizer("ai4bharat/IndicBART")
verify_token_ids(tok)

# --- 2. Compare tokenization efficiency: GPT-2 vs IndicBART ---
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

test_sentences = [
    "यह एक परीक्षण वाक्य है जो हिंदी में लिखा गया है।",
    "The quick brown fox jumps over the lazy dog.",
    "भारत में कई भाषाएँ बोली जाती हैं और हिंदी सबसे अधिक बोली जाने वाली भाषा है।",
]

print("\n--- Tokenization Comparison ---")
print(f"{'Sentence (first 50 chars)':<55} {'GPT-2':>6} {'IndicBART':>10}")
print("-" * 75)
for s in test_sentences:
    g2 = len(gpt2_tok(s, add_special_tokens=False)["input_ids"])
    ib = len(tok(s, add_special_tokens=False)["input_ids"])
    label = s[:50] + ("..." if len(s) > 50 else "")
    print(f"{label:<55} {g2:>6} {ib:>10}")

# --- 3. Assert SEP != EOS ---
assert tok.sep_token_id is not None, "sep_token_id is None!"
assert tok.eos_token_id is not None, "eos_token_id is None!"
assert tok.sep_token_id != tok.eos_token_id, (
    f"FAIL: sep_token_id == eos_token_id == {tok.sep_token_id}"
)
print(f"\n✓ SEP ({tok.sep_token_id}) != EOS ({tok.eos_token_id}) — tokenizer fix verified.")